# 新生成数据长什么样

本文件夹的中等难度模拟数据（种子 2028：噪声 0.12、位移 0.08、幅度 0.35、伸缩 0.2、缺失 8%），和调参用的 wpj baseline 对照着看。

在本文件夹（`zrz/simulation_new`）里从上到下运行。

In [ ]:
# 设置
NEW = '.'                                                         # 本文件夹
BASELINE = '../../wpj/predata/数据准备/outputs/simulated/baseline'   # 对照：调参用的 baseline
SCALE_FLOOR = 0.1                                                 # 与 dtw.py 默认相同

In [ ]:
import csv, sys
from pathlib import Path
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
%matplotlib inline

sys.path.insert(0, '../dtw')
from dtw import prepare_expression, preprocessing_config

plt.rcParams['font.sans-serif'] = ['PingFang SC', 'Heiti SC', 'Arial Unicode MS', 'Noto Sans CJK SC', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

CLASSES = ['early_peak', 'late_peak', 'oscillatory', 'transient_induction', 'transient_repression', 'flat_noise']
NAMES = {'early_peak': '早峰', 'late_peak': '晚峰', 'oscillatory': '周期波动',
         'transient_induction': '短暂上调', 'transient_repression': '短暂下调', 'flat_noise': '平坦'}
COLORS = dict(zip(CLASSES, ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#7f7f7f']))


def read_table(path):
    """gene_id -> list of values (None for blank = missing)."""
    with open(path, encoding='utf-8-sig', newline='') as f:
        rows = list(csv.reader(f))
    times = [float(h[1:]) for h in rows[0][1:]]
    return times, {r[0]: [float(v) if v.strip() else None for v in r[1:]] for r in rows[1:]}


def load(folder):
    folder = Path(folder)
    times, raw = read_table(folder/'expression.csv')
    _, centered = read_table(folder/'processed'/'centered.csv')
    with open(folder/'ground_truth.csv', encoding='utf-8-sig', newline='') as f:
        label = {r['gene_id']: r['true_cluster'] for r in csv.DictReader(f)}
    with open(folder/'gene_parameters.csv', encoding='utf-8-sig', newline='') as f:
        params = {r['gene_id']: r for r in csv.DictReader(f)}
    ids = sorted(centered, key=lambda g: (CLASSES.index(label[g]), g))
    scaled = dict(zip(ids, prepare_expression([centered[g] for g in ids], preprocessing_config(SCALE_FLOOR))))
    return dict(times=times, raw=raw, centered=centered, scaled=scaled, label=label, params=params, ids=ids)


new, base = load(NEW), load(BASELINE)
for name, d in (('新数据', new), ('baseline', base)):
    missing = sum(v is None for g in d['raw'] for v in d['raw'][g])
    print(f"{name}：{len(d['ids'])} 个基因 × {len(d['times'])} 个时间点，原始缺失 {missing} 个格子")

## 1. 新数据：每一类的全部曲线

细线是每个基因（`processed/centered.csv`，已补缺失、减均值），粗线是这一类的平均。

In [ ]:
def class_panels(d, key, title, ylabel):
    fig, axes = plt.subplots(2, 3, figsize=(15, 7), sharex=True, sharey=True)
    for ax, kind in zip(axes.flat, CLASSES):
        genes = [g for g in d['ids'] if d['label'][g] == kind]
        for g in genes:
            ax.plot(d['times'], d[key][g], color=COLORS[kind], alpha=.35, lw=.8)
        mean = [sum(d[key][g][t] for g in genes)/len(genes) for t in range(len(d['times']))]
        ax.plot(d['times'], mean, color='black', lw=2.5)
        ax.set_title(f"{NAMES[kind]}（{len(genes)} 个基因）"); ax.axhline(0, color='#bbb', lw=.6)
    for ax in axes[:, 0]: ax.set_ylabel(ylabel)
    for ax in axes[-1]: ax.set_xlabel('时间（分钟）')
    fig.suptitle(title, fontsize=15); fig.tight_layout(); plt.show()

class_panels(new, 'centered', '新数据：每类全部曲线（细线）与类平均（黑色粗线）', '表达（减均值后）')

## 2. 同一张尺度下：baseline vs 新数据

上排 baseline，下排新数据。新数据的曲线更散、峰的位置更不齐，是 ARI 下降的直接原因。

In [ ]:
fig, axes = plt.subplots(2, 6, figsize=(22, 6.5), sharex=True, sharey=True)
for row, (name, d) in enumerate((('baseline', base), ('新数据', new))):
    for ax, kind in zip(axes[row], CLASSES):
        genes = [g for g in d['ids'] if d['label'][g] == kind]
        for g in genes:
            ax.plot(d['times'], d['centered'][g], color=COLORS[kind], alpha=.35, lw=.8)
        mean = [sum(d['centered'][g][t] for g in genes)/len(genes) for t in range(len(d['times']))]
        ax.plot(d['times'], mean, color='black', lw=2.2)
        ax.axhline(0, color='#bbb', lw=.6)
        if row == 0: ax.set_title(NAMES[kind], fontsize=13)
        if kind == CLASSES[0]: ax.set_ylabel(f'{name}\n表达（减均值后）', fontsize=12)
for ax in axes[-1]: ax.set_xlabel('时间（分钟）')
fig.suptitle('baseline（上） vs 新数据（下）', fontsize=15); fig.tight_layout(); plt.show()

## 3. DTW 实际看到的曲线

`dtw.py` 计算前会再做一次逐基因缩放：除以 max(标准差, 0.1)，把幅度大小统一。这里画的就是缩放后的值，也就是 DTW 真正拿去比较的形状。

In [ ]:
class_panels(new, 'scaled', '新数据：逐基因缩放后（DTW 实际比较的形状）', '缩放后的表达')

## 4. 早峰和晚峰能不能分开

6 类标签里最关键的区分：形状一样、只是出现时间不同。蓝色早峰、橙色晚峰。

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), sharey=True)
for ax, (name, d) in zip(axes, (('baseline', base), ('新数据', new))):
    for kind in ('early_peak', 'late_peak'):
        for g in (g for g in d['ids'] if d['label'][g] == kind):
            ax.plot(d['times'], d['scaled'][g], color=COLORS[kind], alpha=.4, lw=.9)
    ax.set_title(name); ax.set_xlabel('时间（分钟）'); ax.axhline(0, color='#bbb', lw=.6)
axes[0].set_ylabel('缩放后的表达')
axes[1].legend([Line2D([], [], color=COLORS[k], lw=2) for k in ('early_peak', 'late_peak')],
               [NAMES[k] for k in ('early_peak', 'late_peak')], loc='upper center')
fig.suptitle('早峰 vs 晚峰', fontsize=15); fig.tight_layout(); plt.show()

## 5. 全部基因一张热图

每一行是一个基因，按类型排列（左侧色条标出类型），颜色是缩放后的表达：红高蓝低。

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 8), sharey=True)
for ax, (name, d) in zip(axes, (('baseline', base), ('新数据', new))):
    grid = [d['scaled'][g] for g in d['ids']]
    im = ax.imshow(grid, aspect='auto', cmap='RdBu_r', vmin=-2.5, vmax=2.5, interpolation='nearest',
                   extent=(d['times'][0]-5, d['times'][-1]+5, len(grid), 0))
    start = 0
    for kind in CLASSES:
        n = sum(d['label'][g] == kind for g in d['ids'])
        ax.add_patch(plt.Rectangle((d['times'][0]-14, start), 8, n, color=COLORS[kind], clip_on=False))
        if ax is axes[0]: ax.text(d['times'][0]-17, start+n/2, NAMES[kind], ha='right', va='center', fontsize=11)
        ax.axhline(start, color='white', lw=1.5); start += n
    ax.set_title(name); ax.set_xlabel('时间（分钟）'); ax.set_yticks([])
fig.colorbar(im, ax=axes, shrink=.6, label='缩放后的表达')
plt.show()

## 6. 新数据里哪些格子原本是缺失的

黑色 = 原始数据缺失、后来由预处理按时间线性插补的格子。

In [ ]:
fig, ax = plt.subplots(figsize=(7, 8))
mask = [[1 if v is None else 0 for v in new['raw'][g]] for g in new['ids']]
ax.imshow(mask, aspect='auto', cmap='Greys', interpolation='nearest',
          extent=(new['times'][0]-5, new['times'][-1]+5, len(mask), 0))
start = 0
for kind in CLASSES:
    n = sum(new['label'][g] == kind for g in new['ids'])
    ax.text(new['times'][-1]+8, start+n/2, NAMES[kind], va='center', fontsize=11)
    ax.axhline(start, color='#d62728', lw=.8); start += n
ax.set_xlabel('时间（分钟）'); ax.set_yticks([])
ax.set_title(f"新数据缺失位置（共 {sum(map(sum, mask))} 个，占 {sum(map(sum, mask))/(len(mask)*len(mask[0])):.1%}）")
fig.tight_layout(); plt.show()

## 7. 每个基因实际抽到的扰动大小

`gene_parameters.csv` 记录了生成每个基因时实际抽到的位移、时间伸缩和幅度。新数据的分布明显更宽。

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, (col, label) in zip(axes, (('shift', '峰位移（占总时长比例）'), ('gamma', '时间伸缩倍数'),
                                   ('amplitude', '幅度倍数'))):
    for name, d, color in (('baseline', base, '#999999'), ('新数据', new, '#d62728')):
        vals = [float(d['params'][g][col]) for g in d['ids']]
        ax.hist(vals, bins=25, alpha=.55, color=color, label=name)
    ax.set_xlabel(label); ax.set_ylabel('基因数')
axes[0].legend()
fig.suptitle('生成时抽到的扰动：baseline vs 新数据', fontsize=14); fig.tight_layout(); plt.show()